# Task 4: LoRA fine-tune of DINOv3 ViT-7B (cut at 75% depth) — Colab A100

Train labels only; nothing is submitted.

**Before you start**
1. Runtime → **A100 GPU** (80 GB best; 40 GB works with `BS = 16`) + **High-RAM**.
2. 🔑 Secrets: `HF_TOKEN` (DINOv3 licence accepted), Notebook access on.
3. `task4_data.zip` in `MyDrive/cpe342_t4/`.
4. **Run all.** `RUNS` is a queue run in order (the 7B is loaded once). Entry = `<run>[@c<CUT>f<LORA_FROM>]`:
   `full_s7` = recipe default (cut 30, LoRA from 20), seed 7; `val0@c40f28` = cut 40 (all blocks), LoRA on 28-39, scored on scene fold 0.
   Time on A100: c30f20 val ~55 min / full ~65 min; c40f28 ~65 / ~85; c24f12 ~50 / ~60. Each epoch is checkpointed to Drive;
   after a disconnect **Run all** again resumes; finished runs are skipped.
   Cell 6 (`TTA`) scores the l75 LoRA at several image sizes on scene fold 0 and writes test predictions
   at those sizes (~50 min). Every TTA file is skipped if it already exists.
   Cell 7 (`SOUP = True`): model soup. Fold-0 check (~15 min), then one full soup per `SOUP_FULL` entry (~25 min each).
   Queue now: val0 + full_s42 of the logit-adjusted l75 (`@c30f20la`, ~2 h); fold-0 checks: refit-only controls (single l75,
   single c30f10), soup2 with a 2-view head, soup2la, soup3la (~15 min each); full soups soup2 (2-view head only), soup2la, soup3la
   (~35 min each). ~5 h in total; every finished piece is saved, so after a disconnect Run all continues.
   Cell 8 (`SOUP_TTA`, off): the soups at the extra TTA sizes, fold-0 check first. Finished soups / files are skipped,
   so Run all goes straight to the new part.

**Output**: `MyDrive/cpe342_t4/lora7b/preds/` → send the whole `preds` folder back (zip it, it is small).

In [ ]:
# ===== 1. settings
MODEL = "vit_7b_patch16_dinov3.lvd1689m"
CUT = 30            # keep blocks 0..29 (= l75 features), drop the last 10 blocks and the final norm
LORA_FROM = 20      # LoRA on blocks 20..29, blocks 0..19 stay frozen
RANK, ALPHA = 16, 32
SIZE = (224, 416)   # (H, W), patch 16
EPOCHS = 3
BS = 32             # 16 on a 40 GB A100
LR_LORA, LR_HEAD = 1e-4, 1e-3
WD_LORA, WD_HEAD = 0.01, 0.05
SMOOTH = 0.1
TAU = 1.0           # logit-adjustment strength for "la" runs (train class counts only)
SEED = 42
RUNS = ["val0@c30f10", "full_s42@c30f10", "val0@c30f20la", "full_s42@c30f20la"]   # c30f10 runs are done (skipped); kept so LoRA layers
# exist on blocks 10-29 for the soups. "la" = logit-adjusted loss (Menon et al. 2021): CE(logits + TAU * log train_prior), no class
# weights; the raw softmax is then already balanced (uniform target prior), so no post-hoc prior is applied to it.
# "valK" = train on scene folds != K, score fold K; "full" / "full_sN" = all train rows, seed N (default SEED), predict test;
# optional "@cXfY[la]" = keep blocks 0..X-1, LoRA on blocks Y..X-1 (default CUT / LORA_FROM above), "la" = logit-adjusted loss
TTA = False         # cell 6: multi-scale TTA of the l75 LoRA (fold-0 check from the val0 checkpoint + test for TTA_SEEDS)
TTA_SIZES = [(224, 416), (192, 352), (256, 480), (288, 528)]   # (H, W), multiples of 16; first = training size
TTA_SEEDS = [42]    # full-train l75 runs whose *_weights.pt are in the preds folder; add 7 if time allows (~40 min each)
SOUP = True         # cell 7: weight-space soup of LoRA runs with cut 30 (exact average of B@A, head refit on soup features)
# fold-0 check uses the val0 checkpoints of l75 (LoRA 20-29) and c30f10 (LoRA 10-29); full soups use *_weights.pt in the preds folder
SOUP_FULL = {"soup2": [("l75_s42", 20), ("c30f10r16e3_s42", 10)],                     # same pair as the fold-0 check
             "soup2la": [("l75la_s42", 20), ("c30f10r16e3_s42", 10)],                 # l75 replaced by the logit-adjusted run
             "soup3la": [("l75_s42", 20), ("c30f10r16e3_s42", 10), ("l75la_s42", 20)]}
SOUP_VAL = {"single_l75": [("l75", 20)],                                  # control: head refit WITHOUT a soup (is the gain the refit?)
            "single_c30f10": [("c30f10r16e3", 10)],
            "soup2b": [("l75", 20), ("c30f10r16e3", 10)],                  # soup2 again, now also with the 2-view head refit
            "soup2la": [("l75la", 20), ("c30f10r16e3", 10)],
            "soup3la": [("l75", 20), ("c30f10r16e3", 10), ("l75la", 20)]}
HEAD_EPOCHS, HEAD_LR = 5, 5e-4   # head refit on frozen soup features, warm-started from the averaged head
SOUP_TTA = []      # was ["soup2", "soup3"]; multi-scale TTA did not help on the LB (v84a); cell 8 = soup multi-scale TTA
DRIVE_DIR = "/content/drive/MyDrive/cpe342_t4"
FOLD_B64 = "eNpNnQmCHbmOA0lK9z+znxDBLPdM9y+73pIpUVwAkHnvrbnn3DtTZ37/9J2aObfu7f79+/ubc3//nt9vfq/q/Hzq96r7e1X/Xvf7w/u3fr+YW79/fn878z7gnt8vfp968k/V+b2i359+b/29Ir/9varyP9W/D3gf8fvpvev2+8/vv/X7nt8lznlv/V3b77+/D/9d1Xnfc06/t9XvTf27pt/fvk+q9773ze/FN1983xe/e3m//r2p38f+bi9v/X3J+7p6v783b/r98N7/u7T6vXHe/f7e+f7qtwD9Xvb7qrdqNVmn32e8b+/3Vb9beQuTW2fBfp/5ft3vOysf9y7it1L3Xdv701u3zut/a1Rdk5V51/nu/uam3xrXu8R5i9ZvMd61n+zY7+NYgreQLNDNlzf38y78rc77Yd6bzvCq92HvP3OzPvOW/a1X/v5twu8qshpzh9e/NXm//33Y3BjD72I6v8v+db46y/UW6tnJ7yreT+/W3sfU+/B+9/wu5Pd5lUW6uYb3qfW+4rw1iimWq/Szgt9X/V70Xp9riRXm+rmxdzHPTt+1vbV9hvvu4u3I2+Zn3++Nb9/edeQGT76n35e+j367OYd33meyuc+3w++OntFctuj3+YWR1dv0qljQM4LYTL8tfZee38cq+hn27wOf1cbU8+m/O3xf3DmOb1PfC/piBP3ecDv/l9/dnNX35pN9fxf0+yU3/v4+J/Vd87uC91n5pLexb8E6353lz/rPW4n8+v0hFvJuKJ993mVnge67VI7jM7vOmax3et4Pz0LeD+/Hyfa+N7y1eF/zW4ObA/8OVOfC3wW+K8yPb9NPDkaczvWq2w16R0dfk9XvuJpchnf/voVduGzeYCb1zm2sY06O8bv7WPh32uP8Tqz8xNO83+XgPEeYQ8IBfCbb2F3OHf4KF9eHA5tjfodNfcv3+6i3L88Q/IC5uaN39i4m8tb+Yl5saQ5Kzlxs88b1eD4b79S5so6/w+3s6ft9U6z1reLz0hXHcePW48ribAe/mmW93sXvAordzzm8uKXnxPAe78Mn1/Rc+Vu+erbUcc95ecXPFVsft1drFu/jmyvo98FcbmXVOuucPXi72Hj9zmU9/561JwbcXEsOZ77y5r9vkeNW4tHejickZR8uK8jSveVlGd8yx5cTVZ5h3bcBcRv5gEmci01UNuhZRPGbLkLj+9o43Ru3/bb1OYhOfIxpZpPz0zsDvx+eDWXREzziNgmPucAEu8kCZVG1xNjlcBmez+en3hXHsjt3g7uOYb7bb3wyaxkbeFsd53WyLB0XGDvNX+XiE+8mruq+A/62Y3CoMcKYyfu63N0Lre/KcjH3Ge0hSuBPCP7P38W5v296a5+s4cRU3lGP33tblWBViXLjGekXyLOHzzoS/7NIb4V6ml3Nz9fvfledHCB7k3t+m1cVb5L1yvLGOx+OcVY5+9KTayw9VPOFJ+c0BntjOrmoy4GI6byVeqehzG1OruBFhQSyHKfT3FgOUmt67+M67jgfRtoRh5Vve28sb78795hQf+O83xq/5Xz3mkwtfuGt7P6E4RHOJ/71LWzc6o1DOolo78y+Nb2xw9jLiYcwuuV4vjfFwMwp8t0nf/eWouKpy9Rq2Kw41JsDSE70YigO+m7i8hzxe9MLdPGAb00qh6v8zeGosUDvypI/xCRyvvPZiQvvbszCstbZyZN8KfnT8w+5SU/dxcly5Td5QuM53x8qkWOyvKONvWtLghIHNbmYpDVJBQ9bmICYg5+M4GaLTS/ernQTHjnhHMubJU5SwI49SyAxGu71rMevfFz2Py5hPN4JtB139s5k1v9kNztpWHxcvn7wQIm3LFPcd5H0Zv0vh5y1YNWJaZXNvsn14sTiEFMHZKs4/bHtd08dI2r8/DOOXF2y1PiKZNV42s7W4GZJPWOL3kDKjKSSCeCmmckwUk4YonIWKu8nXun53oa1gSBBiFB9Y+LNLb3XVBZm4nfeiS1OfUqUd2yTMk1yl7f9vZZBAkUyU/4VySx+4N1FXHXHSG785/tDLIPXVxzwFDkOfuRg3RRGuFZ8etY1b+xsSrOthI7GRljQ9//lBr87bnxT4U8oQlLEYO8nCXfh53LLRcxNwfH8DQkAx3GS9bxgkyKxOeDThqsznIAXgjp+IblR8daslEky5psvzEkkT0sNmQtLCZRqr6xHY0lmeqypdpKYTHoXv5yfnn10asBJ7Eyi82whS/3SvSGZ7JxMz2ECW85C7tjAHtf4wuckH7x5xzTp8U2gt46iUHjeMbEybyMk3rK6e6Edw8nXJ5CmeDyki4nRRwdHynSLrC1rGpebJc/GpgguIm8qspzLZHGp1PB5xaEYigxf1dglJyd1IdeafPiY7uuSUnklX4zJ5BvvF6yTBuF9mzzj3YCpbjbvYuzFYnTKu9zR86n5MK7mEm5SSNykHMmF3h4+06RASaXx7uUdAXINyhVAi7w+3v9gGE36MHjEoBIm/jHnGGJMML45xkKK0LnodyIqXiC5bPLkJE4npYcASEyHQ0c+c0jWrVVucvwUo6m9cdEnlRb4BmVg+8nJpGKJ5NupGQ+gxQudyaHe2hSFTxKJ5IiJJTmFpwB34kLvYjMdT1fELhLc2FBuIllfCsDcc3wKWVfKguRt+YYceZzwTWaXXE/0oraGfFebaijBMIezxDSSd4INxQYPFcPNx7yFPZRruRPekhDcJCqJGk1CEXcygDTZ+2dbKQvw6kU1SlpPFhkEZSvt7AvVCCU9dVZc6gCL9aFenKBel0UwU02ikEMQS6KGyBXH8JMhPYd7cpIDHJSZNE48d1lbG+do3zjReMbnIpPSZkfvAjSVmjoR/WjUL2IJZ+R4XSuxrNAF4tGxx3N0XCml5SF+4YeaDQmaFkjmxe6ExBRlxTmdwBaH6jz1QGI0BVHOGjBcEiLvP8larDU1xVZVN58dZ4AnOElfxlJNNCP+KVey9Qh4XDAn8YyiTEkFM+xE3nfv5iBJxJNGmm9xQXEbCUFxDMnzg+PlfyuZc6XQFKWMY2sQJ+ygPNWpCZOSBW8EzFlkNDkZ+YOFdi77nT9KvBhkEvcky9TJMdstcd4pyKJk2QZHnZQXFOskD0nNGLfeX52dwH0pCA8r1ZTsd3HUIvXpzQESTpPAl2ZCfkLqiz8D0OJ0FVUI+KSJfwXCzdlLKeg1ru15ipOQEjGJbKl8r4Wp9jwAGqJmh5DMVw9ulG39sL8z1Hz59OLUEY2SolKwpdLBVOJiCxAv775YYMqoGEY8U1PbJIkbtjzQReJyU31T5GNYgV6oxlJWJp20FkvYPtSyrwAxm+nRGi8FX/wwG5IMLCH84LNSEOACj5iwsBSYwiEj1RvHGwb+fZEkDg0QGEQ63gOwe+LW49kCndfRGoNXpxDLZd/UAkNGBHKQgJslAvdNWaFX/erMlKWp6a03bzL+TiEd8Owm709C4Z7hGgsMKMXdmCnxJb6jUhRzaJPiJnWidntHo5sAmaoa4AMU8shmNCXIpYw9JCwJDr4MVPcEZUy5FCsCEuF2kmbgzMm5uWjg43i7hOIhIQHmy2mxqhqBsWAWVopxqp01afLPFM4NShEYqxIBrDRTVDXMRq4L88Y3tzEfgCa5BLXuzeJNamlBqpR4uYFYU9aZVHRKMoVQuTlanGStWS73EQNPDGzyzyKPC0bPR+XAB4V6u4TvS7VqeU1xwC29pUwOE1Du4GqC2wiRpYAY0dBhlUhSN2kHicrF4z5IE5P9xwI8RsCRRaWkeRQx76vRYrsp0U6uMcVFWTuQC/BDrhqwiVI50bFBajfXejDQ6DwXQbiBfswFx6SWlD/gMIgDwSiAFZaYAHGazDlLWiRSkmREoit3F6QiSdngGpvELBgVL6swZlkyiJzAw83nwdJxchr2Cf8bj3+hdK4bsFVugneK9IQfMoqD22oyrKSdhwTRqufuwpEZXoL8kJ5uHZDCJfY9JlNi5UleyoTyLbM5lcBGUtnkM8GfCvo0ni+59/PoFz5uz0bQ8UvU9rglVyRRzdUmT4Nvo9ClBjuAbE0CEVyjyg3H+VM8vuXqAExs25Aq1Sxg2IDyISznkuHGfRZJGHxeKIrWPTf+I3vX+aJzrJliAeawYN05IAV8y98FTMPfpT7M5YPW5/YrVp4SibSx4DqyZuJiAJA5RpwPC8wmiSwWOrawiVoA1heWUmri+ai/IQGB0jH5IUol64Vbif8lUoLV5hglVAOhHNi3wdybPKQkUQbU6uT0Xyp1HCcHArCViNWek1RCSaEsrshhiWT4xMPOE84kEAjkuO+YUmqPhI6Ohz6ACnGtqQ3ANXJlFbcAVow1Fmn54JiXpb61XHiKHhIq7E0gOM41OcrRe5T1I5libiYRWMwA19+kWdmWA8Z3SP06+U/Sxl4w+ySPDZ7diZ6ks1DsLFOTlCUap3y/pLUvxmYrqdQLIAUefPBEAaIqtICxdizUx0yDNQEpjU9M4EuIDzUCL5GIntS7cFewHgBYgcuTw57RBxXgMMVl6qc4lbjpQRYR2q8pmVNRElhxh8gpLsICWE0ia14esiEpWWispr6KVUJTkF0nHAMvC4leMfZ7xHTfGU0BdCGLidpAd4e0yi0pSlpwKYpjpA+pwQw+lOS5yTKlysoE4RAGSEmJ0qLVo3AW+R5qCWPeAF0nbILbhXaDSLssaiqTg3rhEslQq1A//cVgEuKT84Dz5PyQH4ROAe4vkHCPB9Vbw/PmVA7YDaEW58OhICC5+yA9FwirAWgPwBScdcpkjmtOaE6h6NElQQ5czuUSmZv3Ft7vgDMlcbvENyJag/0TB7Kzb685YGAHMXpPCHxzY60tDJiAP0vvFZULaTFub1GbatGoeBKC4KWA2Cj17jQxmMRCXJ+U5FIN3dQlydoGv0UmseBih/OcMoqI4TTsZV4+8lJguyHCUyEkWoDm9sB6Q3fJCMFgiVpnSahUQrDmoEE4Pb/fhL71ngPsfcXCr+cqnw+CRK51oNXPktIArinbcgGRPcRg7tKrEmUS4SGaL3REA2NHftJkQ43vSdYziyg2wKyUsngPcEUgwKti5YBVFmx7YeMFYy5N1NCh1FxwcKWfh4eCfo1p5ZobRxruMmyrWHZCXazhs6MLhA46XMBQVQh6KBVyeDjexe2YlTfA2CVzGs5GkrnlCjHcRIeDuC0ejBQSoQmhcQuMBP8BNAWWjTlfYG+ENk1BI8dMbINBOyJc6JxSy+cqS5N2UwCRDowDqy75spx0yegWCFlqLutgZGDVX7kcotD6LKz2wRjaIjDVZAip2orsA1JKUULiX6iLpCqXnKcAnLKnBZTbst2wfLPCJRgniO3AWmaTBZ0bCBQpVoxcJgK/J16Uc9YrvQpdR5ihjA9Pl6QwniAJsfUBu299kssxRHeZ5CWiFKehQU5gEqC2k7xxDthrjnAKFM4LKerF6PdDwuwDpn7RrCzh8XdQTgMKBvsVcAmsmCXSHVTMQ+CzyA1RM5iDJvsB3q3AnUH5WKLawxHRVE4eRAGaB/ElhRcRRaAKCwRyqVWSj2cNcczXglMRyLtFjiHhghMtlgG82EBobmJiH5I5AsmCBfFQJDxwFGO6SQV5VKdclChK6aD/QrzFVgYQSZYS8vScrZGi0gPgALM/ZK1EkVzTFcXl10FXkl0GLqSOgmmXU5ojxq8T7sWmU48gyziAeij9jvKOhqOJeVl8Eo3VUPBamWNY8iZ7KfNvCAUqu9GpJSkGmkt6xVKSUCbEHNwSdAUQQ7gMWBHzNPWRMF2vUFD52cbyJG8X8WtBAKSYSiFQLAgyxiSZMfvUOTkz8fdF+Eg8ywEcU/IjOnxX9Eo9oDIGuQdMEaqHhxYpYSmpllE+DKkCdNWqx6IuWmkZBP+c70ga8XJc8EAp2oZAeRfmPd4Crjwg0kW+Sn49K6QE2Yc8NFhYwh6VqjmmkfNYN9yzYocDxRRMJL6z5QMJXobH5BnxxuDR2X1IiEQ5DpI6nlSmqABRfYjrtEEKiDr5Ue4nepwr72JqqLoJXOgQrnK6xGTBDADexN8LgXOAhmSyTQ5wguyJf16dAFmGJ4MSZyVKZTKgwDHCE5GYcwQLYzSDkBNVXpJRXD3MMJlvAgfx7bL3ZCHx9mgpxfbbUyvV3akEwf9JjS6JrQJxsYZC7l6NKpEQxyKd+6WVgw8/lq9Q4gkwyEbMrs8nC0kBmuwMb4nqZCBJ1+E3FAUUTEk7RNWTggWih5waSRDOXrFfjtSCfCwbiQ3GgsI/QXaFNFUqCA6SThX5V5L8Im4ElMgyciyAthVzZ9/iEEoOMpKx8x16MpE4h0TNeMJj5h8MEtEKrEbwdPPr6NCsChIq4xmPyWhTY0APJ4CtRht0hWpTlxdKRN7ucqYhE1YCTJp9Kazlq86nxD5Hzd56UKApnBpL36RjYI+qxgDz5BTR6SY9ohqQzwtIDmtqpnQUoygyVRPz5eEDEjtIxHp8W24udW/IqgH8zgehDESGgiQDnhpmFdFs9Gzxd1cQIhePPeKRQUnAIq9qH0vUJp0L0kbqBCxoBacyt9C8lPU2jQ8WvogxqROFIQcDO596CY4ehXXcDVB0Q45RmkTTfj5qeaFHQJhRLCSnW/IEm3sLmFLKwJhx/2pJCPJqExaUvAt0oDxxtxrdNrgT+TLI6gEZXpWw2tuiF8TCr0E343fOVe2Nysa8AGFY8s++qlNxa/2hdYBMp7XCQT2oHhNBScpOuMJBvtuwki2POmr3gFziH/LRlLm0pvTI5124YYhHBLKiLlQEwdWAI8lmBrShkAHS4ZFqLpG01BRB4h0cPAgyfTeoamS4Y0fAa4NUE3qETFIOkZoCR1a0ZlALHAX/I6gZP0nVQqaEVJLsriF7cdCFEIxTfe+matStZZHVf7r3r5uqSbhtY0BfAOQap5Ot2LWB970ooEkmF4AvNLPn3JWiFvd0P0n3oRVlyP8UvSDrIOkiw4yRJ8kBot9kB1L/qOOt3paDSEty2hL8yd8Tsy45ip4Lnjy4LnWetdQtHXAwB1HmAwqbSy3wA7DOMoEj7T8k4ll2pXgj6+r7Ja/BoQBlzlIwaG29iINyhRhqlFRopvyUhFo5aKGIVsrPfu8O1PYJ0NKBeaihSEUB739ARfIO/Fy7aFddEK/zcCRyJW9uQkaL0pDdk2F8OmM1qANaw5ZTmuaScccSL0n87ZoDI6ZFT2sifijISR3H8kAVQekBPAOpAYSAV5PeUjFhEn1smYFcIrMHty+aa1TyQpsdeVxEyHRMHcuDmm232IiK+joQCYovZFIWURzWc2yN2zYHMT70LZwTVmy270awLk49u4QOcPtQBBqQ8EYTkNeebZbg5pTnpp9QqAi3GO1E2cKThpTcDYvQCFCaBqkmvPTYIKTWFIFhbV0YqJh2GVoYERxPC6xJNZa6tPh/tbhEncDvNNgFYhk1ZwAKMD+IpDnD9D4VgqeI7Ukv6wqFQPMUmnIWjnJgOHcUlmQgtAUQsEsKKv5kJJtAvo6ADqCQJBikJRj72CY5NrKBXFBut7wWOgcIi5Gc2bqVyIUCMi5PrRGxc5S5henEfyb8KxEL0lMIk1vF+IG6Qc0LgqT4kEo4aoS2NWxEXadWM0fjChxKzA3hxT36+WTnV/QEfIPaHUeJGEkhGRUzYqH6wwyKwx4ZA30KR0+OQDflITk+NkZznj4wl3rVY+kzIKxJSwAs0AzS41lewuVk0H5ARyHMIqpS8qjUZZGj31abWQqqWaWU4UfahUoMwciR8m06ri7HvGk0A+0/ala2l3L5Rv08vC8rPKtaYHGGxpu7UkESQygNqa6iWOBl0froAfAdNhEPbTNjOR/n1ItS5CiVhR+qXaGFmG8MtFSdU03QeTdXDSWpqspt+/Gm5SHgoq79kmpILlID+/NaIDOe4iBqXOneUaXBwWtTF4kmmKF1yof3U18Z5/FiICgjOKEgJdU/PZEAmH23/ffYL6oS+6wSwyhMA84h9cWUkUsXkqsDdhoIQzpgBHRTK7VyY881oAQdQGi8JiwScDLdktAv6k3kaSlKuC8BTTtbAVdTq8+xCmllKN2KzqjhaNmkGjo0AyMes0vS7slj31Oc/TZbIUWCPQtWsxqt8Et3T9jof+CWC3ymbPJBLl0IDYcmGnr/7vmyvSvW3ajyY6vJgsDV9HXUoZwdxB1ywFLzp+1jQJfRY1Eck771dwDb3jG7WFJaDljKFS9B42qndDxiGGv0uBbPWRO0RWDiCW1Nn89VgEwf0oDerPaq1VCUjUCJC20X5jkrswGEpa0KIrcUD20/BVQ3RdbxFsyNdrrAoR8cLrslYrfJHUQhHwyXqRgThZZk+SirVLitFF91hI1sNBciOrVfO7ElIgQ4F/wX6Cbe8KB4FvEtCxikAVLKNCCRx5Lqqh3VjSKeo9mW3IsGgLl2CpP81mw/GVXNgYol+0o5JG1RoJFAvkC76FdQ+B9FecjfRICVskWmj2AA8ePBs8ZxIzOO7QRQavUldPmQXMKZGUWbbCxw5bjGBbIKZawrorq110BB7RH7JR9E3SInngID9RIIiAzT2YZEZmeoKXAQwwFqA1pFdjBiWraEwiDWYrA0ByI8BkspOyRZy1GtjnVZz4OyNgJ+AGggGlpDaWccPuji0xnaAWgIGjdqqtpxFSDVQIew03E+xXkjFKSgRo1ncxON5WE4wLfx7JQelOilTGGHWvAPMCpZQNkNh5CZtqpRq0MzFihKU66MU0/W3w8mD2wNrTurUYGfbJMPu0wgzIATA0PbwWQJkmoP8Jkm+GUdYCdRXMxikgSLHLzDgJMhgAyU+aEjRKrZrjj86uXOCciFOvcg+JtV0SDZsZIWfpkt3OOHGk0ljIKgLXxNEy04kkg6FWNm49U1yLe2GodUC8uicgBgYZjHQYv0NqGJ9R/mb5QYqDULeEupMbN0vyh10LlIeSbPudu7fBGZaErKMsaiXSE1A3bw64fed5PVIyB2ZmWrUHr3fO3LNAUISCTeq1u25xPFmaNqGgYgqSSqlSC93CDdMPRYCgWg718xFxrARJ2j8iypCuUfTWuIDAriQarXoUAQ3mEG0XG3tR6sONxqcb/quFcnlvuKphabE6mhl+Wiv83tx+YEzI0xrV5zU287vxFy0w8kOQ6OObpIYMy2NfAT4aayKQhcdKGmfAfhX5Ztdv5FzqsrEIi3RHMQFVzbBmyMFjzpWp9Dr9wQjgnL8jNpCwAjRyiYXBWwtOgUUO2FkCYuC4a6YcNFRY1/NprEg0gEMTYqItoLj67in644Jt6Alc5f69HYUCgsODaLbzLlxKZRKRG4I13cpik2LtBVxa46wES3ijAGwpXWe1uw6QVWauUQAJxPIeFCPxI3MxJtR7p4JxW0hT3beldbKhdIhK6vR4KOx7S2iGoyhEJAE9aKsUFUnkuswGDTGFQMRIAfhwm95lA5yqjOrkN4evtCSL3Ve9cys59I32YM0gYGZdAS4Jmyw7JF5I59OYNymOk/Gz6lnQdpCvLpgxqREQxk8zufgt7a6Cgv7TJoLI5Ja9vlrZ4ROQ5lIrblrAgYUvNVWxqTe+el5Tdg5VlHDihDglLEq58opjeksMmC5XDXJp38Bl2Cy0dXHVsIniUXj2KZ0VCXcUAMV3HcF0ikeD8kfqNDRcPUOHtSMMQCyqngo+hRhDAt1X8N815neX4mM52rR24rW4nMFQgfsRvKdIoivhRJwLWrFfBFkLVdkGcFdIc4EMw+WSJ507ZZZztoTFxUxbFA1AXkDzRy9q1tGcgWIrR3pY6iXhB+mmDPLMEHBVUCkpc5KN4aU4RKhUbYS+AfGoDo6aL/1rF3V8zXwRrIHxZfVToT9iXGhjYDt0wLlSBGI0aw5cT6RtzhbpXjJAkbOdmSuw3HdRWhntnZah4P8sIwy/ymKA0uQ6ygreBN6ZGL/GDUcNfZKROjKP3Y3sbkAbPdoS3iOGyjcXNw69ar5npIpbe9RUkLy3qPDe5hTKCspcCU6JOaMMOhaEGmthSYKdAtWkPtQIo4umDeQmedWfVEEjWHG1GMkYGnwr/2wh9w4S71RuIcZIpF3GLOm92TtqiUIEjTEqnpMIEoqRsqbmpeIN2mlY5RirFoW8NGwToSV6Y52Mfm1KDjOCukOBL21AOIy5gCppaabpMtCpZcTyG0rO43BwXCthau2nGHfEI5n28dC6aEZq01L2hWZMlK+QgQ4D9XSi2kv2R3pvYJdqf7eZjcsCOJrnj72FEHsIGUMkXota3bqXpntcgK+RnnU7PjnSBGOXEgOZTFiIDQox0ncRCzMWKCP2KYJjT31wM/qNqcNEfRvdIfZg/y1WMP9WUSHqi/03vKC6HpqhTz2xyyUzQbFvjsZE7wQub2kIE4PlPkgzF9VxFSwGe6qOSwy/5t6rlz7Y9AXwNOAOxgBTNLtKvXWgWUUea4kY5sO3bZBo0KsNxqa51nxZUOMslG8p68w7CNLwJbo/8R1AiinPBKFcDoGGqFozz7OHHvWAVm/0r19PZUIWNA3MlYR47HdRAaFP2iF3SeUIUA/bIPcP/j/hb5xt95NeFVoYFCy5lvTP50P6+TuY79wwGkRuHelPekRAFxczlIwRksMpMq3OCCBOW2ZYIhk4BT+mGaihAgBONKVFH1bLslXYxk5XDA+SroQAYviE+MbYaga0hRjide1qWNOWAdtqXzlwlvRfQuZ5cdFWotPcvopFO9Gkrk2KWoYHhXUogGeclCMtkE1C1FPs5vZHvo+DtIKJxoxXARUK2xoaeodnBiY+m2t8PVldN4DgiPrG/2kCKdNIcJqQqmr1MOx+kmY5SjZR2loHkdspGDGnm1fkKhhGKVfaB2QO/lTikivCLQyDjuypVQ+DgBw06ezb6No3Q60Dxik2Hdjb/qx5oOCPTK4rWXMZvJ4gd9F/UT/YtcEZLSQzPGOK6H6aixf3Qt0n7Ok+rZEQktHwKAgtIKH2SvlnIWZv0WYZRpYHTol61wQ3/DtWeEWQMHf6O845gOO9mpdvn62MJBJL80AtEd8I2yJfkYwl1hfkrak+pfshLU+wwRYXgqAm1mM1nqbY/PtQQcqVZ64rPOyCrlGeyyuYo3uVlnOCSQ6CFoWKd5AxALScaUqFg2hUkuNL727nAUFCT3qFXRwzIIiGoAApv2nJFELlXA9PtdBiDQKaZOUUZfPdY4HtdpakgUpADJQnBPjn7zC5wL6JmV7kwPRe+UKoWLcGdjUwC68qJNNsGPctR51qNAzuqxSVQbHHqbsyEPrkXFYHnYJTJTR9JSZ9Ig8mnN6I0eRcf3LsregCtFhKXoKDCA9eMAsa1wOrWPk9ec0UYsoGhT8uNEQ8aZiD4jzCD1AOIo+rmMTwM9Rj2BWPHu8M/VCl7bfZR6DADPEf46YqXNNAYHDqwm1v5ywLBtg90SekjD8exRA4DcKQG6TmldSZhQAroIpinT22Hu4ZDSazdaS3UgfR+mbNmi7fhp3I5Df7odB+nkP5Fo60PHybZAJe+AXCyKc+ZRAmJ8wkBUKiMWbFvMrPCO+nG7iBr7pMYJBK8qWuTQ0S7XbLed+VCgEUdVKllxk2tcGx2dYACybCcVk4/RNn+dKRITeFtiFZ0bcACX4vXASuqA9b1k3MB7zr6G47y97SrOqdqhLNDUUjM73+t+TQKklwCHzWjxCxvo9DBmHoRRkgtV1WVQdmorVQKUp/1pl67cok1LBUg73ojEMygfpDSQy5fdE0+KCTTonPSUI+264+p3SsV1XO21MdhWUrB09Q4oNo+NNNw6EwOOcz35AmrDw0Dw+cKThsGgUieYoajSKY8tcDn7+qj5BvQxzOWKW+78gEhXGPeu6yGDuNKUDWzogLxZrADNsIDbzkA9krWsZYtBi1vuHDECqecp8laa2rENh2Ee+0dt8z+bUapQmfokTJA8TJBKACRPQaZyto24HQzPcaY4QaDJ/Jlr0YEEzQcgqOeEDWeMr3pjW6OA33ubeeebOeR4dLbDoTT2re48q90X2wfYqkY6AdjohL+SsRXNdVJ7QiOg5TjD3sZKJ5Wj4nIKOIipkk1JGFweQnNlo5SmZQ2reTpwhjRPgsKugQOZDjEXX2pLHqVsExHgjjGge2ycZWyhg/C/0bqMhILnhb1WZE8vI9Ib0m6ljcMAlKS5RG9AmmOxDnltMn7EWD6mbrByigaGBMvAU8IpWyaf+JZnoWA7b+CV4Dvx3n2X0GFfm76xwGglL8Wl+CAHH7PBxGnglMMIlu3HJjk2kYZhYx4fxjlizQcM1l6S2T56RYLGRIoa4gEVszoyh2BS0Dt884j57SMbzhLatFSOebiHCM0UA4CNkXbPMWVr56m0hcNRqOPQw66F2Th3DJZzqjOWgZ8tdOBFW9wseQZDYZpmYiP/6MWFX/aBCA5Yon3/2llBzaTijUGOTFhjipZzCZkkxcSTL01GLMl0k1o3QmN/iWKhSQMUviWpMr1AsXZHb5BznJiiMjvEiMmXg+SBmSLtvHk2j6QXYRd6tDImK8hmekEzdx143HkvFJ/C9koQpJQcfQvK4RwVO0QOJK6jQpzWfXfUmv4w3prhEtXKR3CkCmuKudEMBdjW54YNnraD3XGPRSeRRs6Yg7AE7JbcHg3OO+Rh6FeMD+id84lFnrM58IcVXgUJ42hFGvrOR4co0OLxHqX6YRzky2Bp+5AALGpVm86NXciYQYkwOdfLtsfoOM+YDwW8Oraljo8w2aShNi8em62AmfJaRs/f/7hvhMjiLDsbRrJyhZDypD2Ob3dqlW1EqELh3lAU2/tcnyqkthi2Idxsfef/cXxumRJX72i3Hfu03RM+wwD9hf1aAJA81uLiJ8sGJQ7mtQ+RMtH2CN9+2RYkHs0QUye/M57ARwCctceAtUDB4GLNk3hU8lyh2W8yIFjNbJVdXum1OVFFmgJ1KoMSmLzfIlzL2rOKr6EsHIekAZrbCwg4x1wv4Al99cbqZjC6iHKNYnB8SkHzMF2ARwcI16v+ze0zJcnWIbx3zT5nwKH2CJaBjxYBR8iCdCxc5IVSYLz3VbehY0V7R2Ho/I5VbNJc7GQW/oKJxXfjCyMpcg2Ues4E//RxBdl04fZnY4/snw81YYbF7NA+9wRAKJ5eFK19YNB887+uA+LYgDYutWN+6txPCeATQHw2hiNOD9rNBrfdkU8g17HboRWpGCc8bduhYUyxAhIoPdqxcezyOAalMFdcPuuVvC4VS7oVz04x5ckHBBY6yR0zzewGBtPSUhGgDRK7SUhUbF2rcU4KY3QYTXN8StARWnNqerkoTocoHkJlY7xxCUUeHCDt2ytmBFxC6GimUw6kQFJCAYCQm3kCRADleLTL0OrohjjGGzflc154dhQdmw3vhkyCJwAwBul+1DzkJBhaqngfQAbs5tFCeyEwcHeWOweNqb5ftz06FkenO1Q/DsAxG5butM0LKH9PghhbberuQ6Sg7lUXH1V/4IbSa8pQjuhOYL7rUHMqAEL5aqB8xhf4wipfz/b3Mg/jMKwhyaojpgDCaxyxwN97mC6uYYE9AvtRJi4eH/In6HKyQ9TiYA/WuVemxmkzW9cxkcQx6VXL7V4djg+HkINgOsRQszKO4e4oOgp3+u2Z1bOYgCDLHeWDTrhyTMysEt+m2zkOtNlhMLKS2T0bSO2MPt+k223rvVsw+MAT57JD+UMRwOSktELbslPCpcvos6sdvlLOeIipc2CU9zHym7BfV2RxygfK4MLL0cOUEY5qY9BEMwTbaSujTOb+zSfUFjldpc7oOKe7mHujaMf+w/slclfsnXx9KfOzMwPHrkVhNfQvTOp2tp1SucXTjojghXdjatbYPIESdNSTqzkzg5n0MCAyYAi5D3wzE6GSOjs/Fs0gcxqLk0Mr8lXLOAz5po6wcQhVDiAuvUJUI6hxTdyb4Tk8pZHWJ4XK686u2NtZmg9d+jcfRo2YJHbDQ33ywWNbva136rIQDH4PhVPavOOhPNGzXZQ0ngEJrHbBckDCchwL6/DRYvbWzqjcpnuUX0yWMvSR16jZQSJZ23s7QgZxOGWLK9MbmNO3XRMmuHE1SIcIDHcnZc4+mjOOplCpbe/yAtrIoWg5qJ1hTs7VQtPlcFfs031M6nlo40FmSAphDzBOp2XUm0G18Ty2SvcOBVM3dVU+Bxu6PhNwdvJS7XwBWhe693l8zlJqHwjH0DOeYXaYUtCUxj56hX66mtUwlL2LNs8JfjIA2ZF6jmvmCT/rLTKfjckhY6fbVy2iELrAng5JcMLiN7r8SOnSTFj7OFF6rugAwqFeZJ2HflG4UGEURvSfWj0oTxGwautr8kVRYl6a9gO5GsWFDhpn8EbhmJFnw5sx36w2ORYmt4eSJm/n5QOxMjYl0+EYq39tzqPVCH1CqyecnY3CtAU0cM7OKPakZP6dJea5qF51mjz4jvVrykf1cuMcexBcpq3zsAbR7GNzuFCdTdHHRiGKrDb/bLaCPgDhDB5tQl+0z6CirNznb4m/RjpMVwWnH5ilGbSG3NRhYPDWxWUyItB+zFK9fjWZY/Ni2bwIwWGlvs2ZiOMZjgMpggCKdvuGpiFJa5lbs1T1z6g3dmI0TzjhIV/bcQJthbPF0/KwCCeNMq3E8h9pGWIdJ11dC3wanK3d4Gbz5KRS38UAdciXFMsYwz07D3UUGuGLh5bMROXeSSYQbdtyQV+QgxgY2FoIrM6froJBaM5V45loMMOz6voPvYcdWbC3eMjedXDB1Pccox0/PFKD9w/2GJ+KiR+nQ0bVHnVW/40n3V5hxsnxUKjymTu1zxLr+gydSpTxsdAsPGxO/WyNjzIDiZ11Q+2zl5Rb9j45z1MO2OuMVhVDtVll77CcHXuqoIvpaj5v88A/ZVAPmcFOD7QQ+Xw23fI7dslpBbRiQ9Om4D47Aymd+1yxE1guod3eqHUdPI6CnlMUED5ymafqzXYHKw0ANap9xGc7iYtSYnza42FMeSPHQb6WJfsHQqIPzg=="   # SGKF scene folds (t4/research/groupcv/sgkf_sig8_fold.npy), train.csv row order

In [ ]:
# ===== 2. GPU, packages, Drive, secrets
import os, subprocess, torch, importlib.metadata as im
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
try: tv = im.version("timm")
except im.PackageNotFoundError: tv = None
if tv != "1.0.30": subprocess.run(["pip", "install", "-q", "timm==1.0.30"], check=True)
import timm, timm.data
from google.colab import drive, userdata
drive.mount("/content/drive")
class SetupError(Exception): pass
try: os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e: raise SetupError(f"add Colab secret HF_TOKEN and turn on Notebook access [{type(e).__name__}]") from None
OUT = f"{DRIVE_DIR}/lora7b/preds"; CK = f"{DRIVE_DIR}/lora7b/ckpt"; os.makedirs(OUT, exist_ok=True); os.makedirs(CK, exist_ok=True)
print("timm", timm.__version__, "| out", OUT)

In [ ]:
# ===== 3. data -> local SSD -> uint8 arrays (240x448 cache), labels, scene folds
import glob, zipfile, time, base64, zlib, json, math, numpy as np, pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
DATA = "/content/data"; os.makedirs(DATA, exist_ok=True)
if not glob.glob(f"{DATA}/**/train.csv", recursive=True):
    z = f"{DRIVE_DIR}/task4_data.zip"
    if not os.path.exists(z): raise SetupError(f"upload task4_data.zip to {z}")
    zipfile.ZipFile(z).extractall(DATA)
ROOT = [os.path.dirname(p) for p in glob.glob(f"{DATA}/**/train.csv", recursive=True)][0]
train = pd.read_csv(f"{ROOT}/train.csv"); test = pd.read_csv(f"{ROOT}/test.csv")
CLASSES = sorted(train["class"].unique()); K = len(CLASSES)
Y = torch.tensor(train["class"].map({c: i for i, c in enumerate(CLASSES)}).values)
FOLD = np.frombuffer(zlib.decompress(base64.b64decode(FOLD_B64)), dtype=np.int8).copy()
assert len(FOLD) == len(train) == 31546 and list(train["id"].astype(str)[:3]) == ["su7tkav", "shlk51f", "sbdqvs0"], "train.csv order changed"
pd.DataFrame({"id_img": test["id"]}).to_csv(f"{OUT}/test_ids.csv", index=False)   # row order of every test file
def load_all(folder, names):
    read = lambda i: np.asarray(Image.open(f"{folder}/{i}.jpg").convert("RGB").resize((448, 240)))
    with ThreadPoolExecutor(32) as ex: return torch.from_numpy(np.stack(list(ex.map(read, names, chunksize=64))))
t0 = time.time()
X = {"train": load_all(f"{ROOT}/train", train["id"].astype(str).values), "test": load_all(f"{ROOT}/test", test["id"].astype(str).values)}
print({k: tuple(v.shape) for k, v in X.items()}, "classes", CLASSES, "folds", np.bincount(FOLD - FOLD.min()), f"{time.time()-t0:.0f}s")

In [ ]:
# ===== 4. model: 7B cut at block CUT, frozen bf16 base + LoRA (fp32) on blocks LORA_FROM..CUT-1, fp32 head
import torch.nn as nn, torch.nn.functional as F, gc
from torchvision.ops import roi_align
t0 = time.time()
model = timm.create_model(MODEL, pretrained=True, num_classes=0, img_size=SIZE, dynamic_img_size=True)
cfg = timm.data.resolve_model_data_config(model)
def parse(r):   # "full_s7@c40f28" -> ("full_s7", 40, 28, False); "val0@c30f20la" -> ("val0", 30, 20, True)
    n, _, rc = r.partition("@"); la = rc.endswith("la"); rc = rc[:-2] if la else rc
    c, f = (CUT, LORA_FROM) if not rc else map(int, rc.lstrip("c").split("f"))
    return n, c, f, la
SPECS = [parse(r) for r in RUNS]; NB = len(model.blocks)
assert all(0 <= f < c <= NB for _, c, f, _ in SPECS), f"bad @cXfY (model has {NB} blocks)"
MAXCUT = max(c for _, c, _, _ in SPECS); MINF = min(f for _, _, f, _ in SPECS)
ALL = list(model.blocks)[:MAXCUT]; model.blocks = nn.ModuleList(ALL); model.norm = nn.Identity()
model = model.to(torch.bfloat16).to("cuda").eval()
for p in model.parameters(): p.requires_grad_(False)
gc.collect(); torch.cuda.empty_cache()

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device="cuda")); self.B = nn.Parameter(torch.zeros(base.out_features, r, device="cuda"))
        self.reset()
    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A.to(x.dtype)), self.B.to(x.dtype)) * self.scale
LORA = {i: [] for i in range(MINF, MAXCUT)}   # block index -> its LoRA layers
for i in range(MINF, MAXCUT):
    blk = ALL[i]
    for name, mod in list(blk.named_modules()):
        for cn, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                l = LoRALinear(child, RANK, ALPHA); setattr(mod, cn, l); LORA[i].append(l)
from timm.layers.config import set_reentrant_ckpt
set_reentrant_ckpt(False)   # non-reentrant: block LORA_FROM gets a no-grad input, LoRA grads must still flow
model.set_grad_checkpointing(True)
NP = model.num_prefix_tokens; D = model.embed_dim
MEAN = torch.tensor(cfg["mean"], device="cuda").view(1, 3, 1, 1); STD = torch.tensor(cfg["std"], device="cuda").view(1, 3, 1, 1)
head = nn.Sequential(nn.LayerNorm(2 * D), nn.Dropout(0.1), nn.Linear(2 * D, K)).cuda()
def activate(cut, f):   # use blocks 0..cut-1, train LoRA of blocks f..cut-1; all other LoRA layers are zero (B = 0) and frozen
    model.blocks = nn.ModuleList(ALL[:cut])
    for ls in LORA.values():
        for l in ls: l.reset(); l.A.requires_grad_(False); l.B.requires_grad_(False)
    act = [l for i in range(f, cut) for l in LORA[i]]
    for l in act: l.A.requires_grad_(True); l.B.requires_grad_(True)
    return act
print(f"{MAXCUT} blocks kept | LoRA on blocks {MINF}-{MAXCUT-1}, {sum(len(v) for v in LORA.values())} layers | "
      f"prefix {NP} | GPU {torch.cuda.memory_allocated()/1e9:.1f} GB | {time.time()-t0:.0f}s")

def embed(x):   # x: float (B,3,H,W) in [0,1] -> [CLS, mean patch] fp32
    t = model.forward_features(((x - MEAN) / STD).to(torch.bfloat16))
    return torch.cat([t[:, 0], t[:, NP:].mean(1)], 1).float()

def augment(u8):   # u8: (B,240,448,3) uint8 on GPU -> random resized crop + flip + colour jitter, (B,3,H,W) float
    x = u8.permute(0, 3, 1, 2).float().div_(255); B, _, h, w = x.shape
    area = torch.empty(B, device="cuda").uniform_(0.5, 1.0); ar = torch.exp(torch.empty(B, device="cuda").uniform_(-0.15, 0.15))
    cw = (w * torch.sqrt(area * ar)).clamp(max=w); ch = (h * torch.sqrt(area / ar)).clamp(max=h)
    x0 = torch.rand(B, device="cuda") * (w - cw); y0 = torch.rand(B, device="cuda") * (h - ch)
    boxes = torch.stack([torch.arange(B, device="cuda").float(), x0, y0, x0 + cw, y0 + ch], 1)
    x = roi_align(x, boxes, output_size=SIZE, spatial_scale=1.0, sampling_ratio=2, aligned=True)
    flip = torch.rand(B, device="cuda") < 0.5; x[flip] = x[flip].flip(3)
    j = lambda lo, hi: torch.empty(B, 1, 1, 1, device="cuda").uniform_(lo, hi)
    x = x * j(0.8, 1.2)                                                     # brightness
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * j(0.8, 1.2) + m      # contrast
    g = x.mean(1, keepdim=True); x = (x - g) * j(0.8, 1.2) + g              # saturation
    return x.clamp_(0, 1)

def plain(u8, view):
    x = u8.permute(0, 3, 1, 2).float().div_(255)
    x = F.interpolate(x, size=SIZE, mode="bicubic", align_corners=False, antialias=True).clamp_(0, 1)
    return x.flip(3) if view == "hflip" else x

In [ ]:
# ===== 5. train / predict
@torch.no_grad()
def predict(Xs, view, bs=64):
    model.eval(); head.eval(); out = []
    for i in range(0, len(Xs), bs):
        out.append(torch.softmax(head(embed(plain(Xs[i:i+bs].cuda(non_blocking=True), view))), 1).cpu())
    return torch.cat(out).numpy()

def run(name, tr_idx, seed, cut, f, var, la=False):
    torch.manual_seed(seed); np.random.seed(seed)
    act = activate(cut, f); lora_params = [p for l in act for p in (l.A, l.B)]
    print(f"[{name}] blocks 0-{cut-1}, LoRA {f}-{cut-1}: {len(act)} layers, {sum(p.numel() for p in lora_params)/1e6:.1f}M params", flush=True)
    for m in head.modules():
        if hasattr(m, "reset_parameters"): m.reset_parameters()
    opt = torch.optim.AdamW([{"params": lora_params, "lr": LR_LORA, "weight_decay": WD_LORA},
                             {"params": head.parameters(), "lr": LR_HEAD, "weight_decay": WD_HEAD}])
    spe = len(tr_idx) // BS; total = spe * EPOCHS; warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))
    cnt = torch.bincount(Y[tr_idx], minlength=K).float(); cw = 1 / torch.sqrt(cnt + 1); cw = (cw / cw.mean()).cuda()
    if la:   # logit adjustment: train on logits + TAU * log(prior); predict with the raw logits (balanced)
        lp = (TAU * torch.log(cnt / cnt.sum())).cuda(); ce = nn.CrossEntropyLoss(label_smoothing=SMOOTH); lossf = lambda z, t: ce(z + lp, t)
        print(f"[{name}] logit-adjusted loss, TAU {TAU}, no class weights", flush=True)
    else: lossf = nn.CrossEntropyLoss(weight=cw, label_smoothing=SMOOTH)
    ck = f"{CK}/{var}_{name}.pt"; ep0 = 0
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cuda", weights_only=False)   # own checkpoint
        assert len(s["lora"]) == len(act)
        for l, (a, b) in zip(act, s["lora"]): l.A.data.copy_(a); l.B.data.copy_(b)
        head.load_state_dict(s["head"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); ep0 = s["epoch"]
        print(f"[{name}] resumed after epoch {ep0}")
    log = {"name": name, "epochs": []}
    for ep in range(ep0, EPOCHS):
        model.train(); head.train(); g = torch.Generator().manual_seed(seed + ep)
        perm = torch.from_numpy(tr_idx)[torch.randperm(len(tr_idx), generator=g)]
        t0 = time.time(); run_loss = 0.0
        for st in range(spe):
            b = perm[st * BS:(st + 1) * BS]
            x = augment(X["train"][b].cuda(non_blocking=True)); yb = Y[b].cuda()
            loss = lossf(head(embed(x)), yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_params + list(head.parameters()), 1.0); opt.step(); sched.step()
            run_loss += loss.item()
            if st % 100 == 0 or st == spe - 1:
                el = time.time() - t0; eta = el / (st + 1) * (spe - st - 1)
                print(f"[{name}] ep {ep+1}/{EPOCHS} step {st+1}/{spe} loss {run_loss/(st+1):.4f} | {el/60:.1f} min, epoch ETA {eta/60:.1f} min"
                      f" | GPU peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB", flush=True)
        log["epochs"].append({"epoch": ep + 1, "loss": run_loss / spe, "min": round((time.time() - t0) / 60, 1)})
        torch.save({"lora": [(l.A.data, l.B.data) for l in act], "head": head.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "epoch": ep + 1}, ck + ".tmp"); os.replace(ck + ".tmp", ck)
    return log, act

from sklearn.metrics import f1_score
for name, cut, f, la in SPECS:
    VAR = ("l75" if (cut, f, RANK, EPOCHS) == (30, 20, 16, 3) else f"c{cut}f{f}r{RANK}e{EPOCHS}") + ("la" if la else "")   # recipe id in file names
    if name.startswith("full"):
        seed = int(name.split("_s")[1]) if "_s" in name else SEED; tag = f"v4f_lora7b_{VAR}_s{seed}"; tr_idx = np.arange(len(FOLD))
    elif name.startswith("val"):
        vk = int(name[3:]); assert vk in set(FOLD.tolist()), f"no scene fold {vk}"
        seed = SEED; tag = f"lora7b_{VAR}_val{vk}"; tr_idx = np.where(FOLD != vk)[0]
    else: raise SetupError(f"unknown run name {name!r}: use valK, full or full_sN")
    if os.path.exists(f"{OUT}/{tag}_log.json"): print("skip (done)", name); continue
    print(f"=== {name}: seed {seed}, {len(tr_idx)} train rows, tag {tag}", flush=True)
    log, act = run(name, tr_idx, seed, cut, f, VAR, la)
    if name.startswith("val"):
        va = np.where(FOLD == vk)[0]; P = {v: predict(X["train"][va], v) for v in ("plain", "hflip")}
        for v, p in P.items(): np.save(f"{OUT}/{tag}_oof_{v}.npy", p)
        np.save(f"{OUT}/{tag}_val_idx.npy", va); yv = Y[va].numpy()
        log["val_f1"] = {v: round(f1_score(yv, p.argmax(1), average="macro"), 5) for v, p in P.items()}
        log["val_f1"]["tta"] = round(f1_score(yv, (P["plain"] + P["hflip"]).argmax(1), average="macro"), 5)
        print(f"[{name}] held-out scene fold {vk} ({len(va)} rows) macro-F1 {log['val_f1']}", flush=True)
    else:
        for v in ("plain", "hflip"): np.save(f"{OUT}/{tag}_test_full_{v}.npy", predict(X["test"], v))
        torch.save({"lora": [(l.A.data.cpu(), l.B.data.cpu()) for l in act], "head": head.state_dict()}, f"{OUT}/{tag}_weights.pt")
    json.dump(log, open(f"{OUT}/{tag}_log.json", "w"), indent=1); print("saved", tag, flush=True)
print("done - nothing submitted")

In [ ]:
# ===== 6. multi-scale TTA for the l75 LoRA (cut 30, LoRA 20-29): fold-0 check, then test predictions per size
@torch.no_grad()
def predict_sz(Xs, view, size, bs=32):
    model.eval(); head.eval(); out = []
    for i in range(0, len(Xs), bs):
        x = Xs[i:i+bs].cuda(non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
        x = F.interpolate(x, size=size, mode="bicubic", align_corners=False, antialias=True).clamp_(0, 1)
        out.append(torch.softmax(head(embed(x.flip(3) if view == "hflip" else x)), 1).cpu())
    return torch.cat(out).numpy()
def load_l75(path):   # LoRA order = blocks 20..29 in named_modules order, as saved by every l75 run
    act = activate(30, 20); s = torch.load(path, map_location="cuda", weights_only=False)
    assert len(s["lora"]) == len(act), (len(s["lora"]), len(act))
    for l, (a, b) in zip(act, s["lora"]): l.A.data.copy_(a); l.B.data.copy_(b)
    head.load_state_dict(s["head"]); print("loaded", path, flush=True)
if TTA:
    assert MINF <= 20 and MAXCUT >= 30, "TTA needs LoRA layers on blocks 20-29: keep RUNS within cut >= 30, LoRA from <= 20"
    ck0 = next((q for q in (f"{CK}/l75_val0.pt", f"{CK}/val0.pt") if os.path.exists(q)), None)
    if ck0 is None: print("no l75 val0 checkpoint in", CK, "-> fold-0 TTA check skipped")
    else:
        load_l75(ck0); va = np.where(FOLD == 0)[0]; yv = Y[va].numpy(); Xv = X["train"][va]; got = {}
        for sz in TTA_SIZES:
            fn = f"{OUT}/lora7b_l75_val0_tta_{sz[0]}x{sz[1]}.npy"
            if not os.path.exists(fn): np.save(fn, (predict_sz(Xv, "plain", sz) + predict_sz(Xv, "hflip", sz)) / 2)
            got[sz] = np.load(fn); pr = got[sz].argmax(1)
            print(f"[tta fold0] {sz}: macro-F1 {f1_score(yv, pr, average='macro'):.5f}, errors {(pr != yv).sum()}", flush=True)
        pr = np.mean([np.log(np.clip(v, 1e-9, 1)) for v in got.values()], 0).argmax(1)
        print(f"[tta fold0] all sizes (geo-mean): macro-F1 {f1_score(yv, pr, average='macro'):.5f}, errors {(pr != yv).sum()}"
              "  (training size alone should match val0: .99115 / 42 errors)", flush=True)
    for sd in TTA_SEEDS:
        w = f"{OUT}/v4f_lora7b_l75_s{sd}_weights.pt"
        if not os.path.exists(w): print("missing", w, "-> skipped"); continue
        todo = [(sz, v) for sz in TTA_SIZES[1:] for v in ("plain", "hflip")
                if not os.path.exists(f"{OUT}/v4f_lora7b_l75_s{sd}_test_full_{v}_{sz[0]}x{sz[1]}.npy")]
        if not todo: print("seed", sd, "TTA done"); continue
        load_l75(w)
        for sz, v in todo:
            t0 = time.time(); np.save(f"{OUT}/v4f_lora7b_l75_s{sd}_test_full_{v}_{sz[0]}x{sz[1]}.npy", predict_sz(X["test"], v, sz))
            print(f"[tta test] seed {sd} {sz} {v}: {time.time()-t0:.0f}s", flush=True)
print("TTA cell done")

In [ ]:
# ===== 7. model soup: average the LoRA weight deltas of several cut-30 runs, then refit the head on the soup features
# B@A of each run is kept exactly: A_soup = [A_1; A_2; ...], B_soup = [B_1, B_2, ...] / n  ->  B_soup @ A_soup = mean_i(B_i @ A_i)
# (averaging A and B separately would be wrong: LoRA factors of different runs live in unrelated bases).
import copy
def soup_load(parts):   # parts: [(checkpoint path, lora_from)], all cut 30
    fmin = min(f for _, f in parts); act = activate(30, fmin); n = len(LORA[29]); w = 1 / len(parts)
    for l in act: l.A.requires_grad_(False); l.B.requires_grad_(False)
    got = {}; heads = []
    for path, f in parts:
        st = torch.load(path, map_location="cuda", weights_only=False); heads.append(st["head"])
        assert len(st["lora"]) == n * (30 - f), (path, len(st["lora"]), n * (30 - f))
        for k, (a, b) in enumerate(st["lora"]): got.setdefault((f + k // n, k % n), []).append((a.float(), b.float()))
    for i in range(fmin, 30):
        for j, l in enumerate(LORA[i]):
            pc = got.get((i, j))
            if pc: l.A = nn.Parameter(torch.cat([a for a, _ in pc], 0), requires_grad=False); l.B = nn.Parameter(torch.cat([b for _, b in pc], 1) * w, requires_grad=False)
    hs = copy.deepcopy(heads[0])
    for k in hs: hs[k] = sum(h[k].float() for h in heads) / len(heads)
    print("soup of", [os.path.basename(p) for p, _ in parts], flush=True); return hs
def soup_reset():   # back to plain rank-RANK LoRA layers (B = 0)
    for ls in LORA.values():
        for l in ls:
            l.A = nn.Parameter(torch.empty(RANK, l.base.in_features, device="cuda")); l.B = nn.Parameter(torch.zeros(l.base.out_features, RANK, device="cuda")); l.reset()
@torch.no_grad()
def embed_all(Xs, view, bs=64):
    model.eval(); return torch.cat([embed(plain(Xs[i:i+bs].cuda(non_blocking=True), view)).cpu() for i in range(0, len(Xs), bs)])
def new_head(state):
    h = nn.Sequential(nn.LayerNorm(2 * D), nn.Dropout(0.1), nn.Linear(2 * D, K)).cuda(); h.load_state_dict(state); return h.eval()
@torch.no_grad()
def head_proba(h, E): h.eval(); return torch.cat([torch.softmax(h(E[i:i+4096].cuda()), 1).cpu() for i in range(0, len(E), 4096)]).numpy()
def fit_head(E, y, state):   # same loss as LoRA training (class weights 1/sqrt(count), label smoothing), features fixed
    torch.manual_seed(SEED); h = new_head(state); h.train(); E = E.cuda(); y = y.cuda(); bs = 512
    cnt = torch.bincount(y, minlength=K).float(); cw = 1 / torch.sqrt(cnt + 1); cw = cw / cw.mean()
    lf = nn.CrossEntropyLoss(weight=cw, label_smoothing=SMOOTH); opt = torch.optim.AdamW(h.parameters(), lr=HEAD_LR, weight_decay=WD_HEAD)
    steps = HEAD_EPOCHS * math.ceil(len(E) / bs); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for ep in range(HEAD_EPOCHS):
        perm = torch.randperm(len(E), device="cuda")
        for i in range(0, len(E), bs):
            b = perm[i:i+bs]; opt.zero_grad(); lf(h(E[b]), y[b]).backward(); opt.step(); sch.step()
    return h.eval()
if SOUP:
    assert MINF <= 10 and MAXCUT >= 30, "soup needs LoRA layers on blocks 10-29: keep a run @c30f10 in RUNS"
    vck = lambda r: next((q for q in ((f"{CK}/l75_val0.pt", f"{CK}/val0.pt") if r == "l75" else (f"{CK}/{r}_val0.pt",)) if os.path.exists(q)), None)
    # heads per soup: avghead (averaged), refit (head refit on plain train features), refit2v (refit on plain + hflip features)
    for sname, spec in SOUP_VAL.items():
        tag = f"lora7b_{sname}_val0"; parts = [(vck(r), f) for r, f in spec]
        if any(p is None for p, _ in parts): print("val0 checkpoints missing for", sname, spec, "-> fold-0 check skipped"); continue
        if os.path.exists(f"{OUT}/{tag}_log.json"): print("skip (done)", tag); continue
        t0 = time.time(); hs = soup_load(parts)
        va = np.where(FOLD == 0)[0]; tr = np.where(FOLD != 0)[0]; yv = Y[va].numpy()
        Ev = {v: embed_all(X["train"][va], v) for v in ("plain", "hflip")}
        Ep = embed_all(X["train"][tr], "plain"); h1 = fit_head(Ep, Y[tr], hs)
        Eh = embed_all(X["train"][tr], "hflip"); h2 = fit_head(torch.cat([Ep, Eh]), torch.cat([Y[tr], Y[tr]]), hs); del Ep, Eh
        log = {"name": tag, "parts": spec}
        for hn, h in (("avghead", new_head(hs)), ("refit", h1), ("refit2v", h2)):
            P = {v: head_proba(h, Ev[v]) for v in Ev}; t = f"{tag}_{hn}"
            for v, p in P.items(): np.save(f"{OUT}/{t}_oof_{v}.npy", p)
            pr = (P["plain"] + P["hflip"]).argmax(1)
            log[hn] = {"f1": round(f1_score(yv, pr, average="macro"), 5), "errors": int((pr != yv).sum())}
            print(f"[soup fold0 {sname}] {hn}: macro-F1 {log[hn]['f1']}, errors {log[hn]['errors']}"
                  "  (compare: l75 .99115/42, c30f10 .99172/40, soup2 refit .9937/30)", flush=True)
        np.save(f"{OUT}/{tag}_val_idx.npy", va); json.dump(log, open(f"{OUT}/{tag}_log.json", "w"), indent=1)
        del Ev; gc.collect(); print(f"fold-0 soup {sname} {time.time()-t0:.0f}s", flush=True)
    for name, spec in SOUP_FULL.items():
        tag = f"v4f_lora7b_{name}_s42"; t2 = f"v4f_lora7b_{name}hv2_s42"; ta = f"v4f_lora7b_{name}avghead_s42"
        if os.path.exists(f"{OUT}/{t2}_test_full_hflip.npy"): print("skip (done)", name); continue
        parts = [(f"{OUT}/v4f_lora7b_{r}_weights.pt", f) for r, f in spec]
        miss = [p for p, _ in parts if not os.path.exists(p)]
        if miss: print("missing", miss, "->", name, "skipped"); continue
        t0 = time.time(); hs = soup_load(parts); have1 = os.path.exists(f"{OUT}/{tag}_test_full_hflip.npy")   # keep submitted files as they are
        Ep = embed_all(X["train"], "plain"); h1 = None if have1 else fit_head(Ep, Y, hs)
        Eh = embed_all(X["train"], "hflip"); h2 = fit_head(torch.cat([Ep, Eh]), torch.cat([Y, Y]), hs); del Ep, Eh
        h_avg = new_head(hs)
        for v in ("plain", "hflip"):
            E = embed_all(X["test"], v); np.save(f"{OUT}/{t2}_test_full_{v}.npy", head_proba(h2, E))
            if not have1:
                np.save(f"{OUT}/{tag}_test_full_{v}.npy", head_proba(h1, E)); np.save(f"{OUT}/{ta}_test_full_{v}.npy", head_proba(h_avg, E))
        if not have1: torch.save({"head_refit": h1.state_dict(), "head_avg": hs, "parts": spec}, f"{OUT}/{tag}_head.pt")
        torch.save({"head_refit2v": h2.state_dict(), "parts": spec}, f"{OUT}/{t2}_head.pt")
        print(f"saved {name} ({time.time()-t0:.0f}s)", flush=True)
    soup_reset()
print("soup cell done")

In [ ]:
# ===== 8. multi-scale TTA for the soups (refit head): fold-0 check, then test predictions at TTA_SIZES[1:]
if SOUP_TTA:
    va = np.where(FOLD == 0)[0]; yv = Y[va].numpy(); Xv = X["train"][va]; extra = TTA_SIZES[1:]
    todo = [sz for sz in extra if not os.path.exists(f"{OUT}/lora7b_soup2_val0_refit_tta_{sz[0]}x{sz[1]}.npy")]
    hv = f"{OUT}/lora7b_soup2_val0_head.pt"
    if todo:
        ck_l75 = next((q for q in (f"{CK}/l75_val0.pt", f"{CK}/val0.pt") if os.path.exists(q)), None)
        hs = soup_load([(ck_l75, 20), (f"{CK}/c30f10r16e3_val0.pt", 10)])
        if os.path.exists(hv): st = torch.load(hv, map_location="cuda", weights_only=False)
        else:
            tr = np.where(FOLD != 0)[0]; st = fit_head(embed_all(X["train"][tr], "plain"), Y[tr], hs).state_dict(); torch.save(st, hv)
        head.load_state_dict(st)
        for sz in todo:
            t0 = time.time(); np.save(f"{OUT}/lora7b_soup2_val0_refit_tta_{sz[0]}x{sz[1]}.npy", (predict_sz(Xv, "plain", sz) + predict_sz(Xv, "hflip", sz)) / 2)
            print(f"[soup tta fold0] {sz} {time.time()-t0:.0f}s", flush=True)
    lg = lambda p: np.log(np.clip(p, 1e-9, 1))
    got = {TTA_SIZES[0]: (np.load(f"{OUT}/lora7b_soup2_val0_refit_oof_plain.npy") + np.load(f"{OUT}/lora7b_soup2_val0_refit_oof_hflip.npy")) / 2}
    for sz in extra: got[sz] = np.load(f"{OUT}/lora7b_soup2_val0_refit_tta_{sz[0]}x{sz[1]}.npy")
    for sz, p in got.items(): pr = p.argmax(1); print(f"[soup tta fold0] {sz}: macro-F1 {f1_score(yv, pr, average='macro'):.5f}, errors {(pr != yv).sum()}", flush=True)
    pr = np.mean([lg(p) for p in got.values()], 0).argmax(1)
    print(f"[soup tta fold0] all sizes: macro-F1 {f1_score(yv, pr, average='macro'):.5f}, errors {(pr != yv).sum()}  (224x416 alone: .9937 / 30)", flush=True)
    for name in SOUP_TTA:
        tag = f"v4f_lora7b_{name}_s42"; hp = f"{OUT}/{tag}_head.pt"
        todo = [(sz, v) for sz in extra for v in ("plain", "hflip") if not os.path.exists(f"{OUT}/{tag}_test_full_{v}_{sz[0]}x{sz[1]}.npy")]
        if not todo: print(name, "TTA done"); continue
        if not os.path.exists(hp): print("missing", hp, "-> run cell 7 first"); continue
        hd = torch.load(hp, map_location="cuda", weights_only=False)
        soup_load([(f"{OUT}/v4f_lora7b_{r}_weights.pt", f) for r, f in hd["parts"]]); head.load_state_dict(hd["head_refit"])
        for sz, v in todo:
            t0 = time.time(); np.save(f"{OUT}/{tag}_test_full_{v}_{sz[0]}x{sz[1]}.npy", predict_sz(X["test"], v, sz))
            print(f"[soup tta test] {name} {sz} {v}: {time.time()-t0:.0f}s", flush=True)
    soup_reset()
print("soup TTA cell done")

In [ ]:
# ===== 9. zip the preds folder for sending back
import shutil; print(shutil.make_archive(f"{DRIVE_DIR}/lora7b_preds", "zip", OUT))